In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcryff.trainer import ThermodynamicTrainer
from imolcryff.trainer.dmff_utils import parser_dmffyaml
from imolcryff.trainer.loss import loss_thermodynamicperturbation
from functools import partial

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/mdtraj/geometry/order.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/srak/apl/iMolCRAFT/DMFF/dmff/admp/qeq.py:33: UserWarning: jaxopt not found, QEQ cannot be used.
  warnings.warn("jaxopt not found, QEQ cannot be used.")
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charges", 'VsiteForce/weight'],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [6]:
trainer.ffparams["NonbondedForce"]["charges"].shape

(22,)

In [7]:
from imolcryff.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[12].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[13].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[14].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[15].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[16].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[17].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[18].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[19].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[20].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[21].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9,10,11]], target_charges=[0.0])
    return ffparams

In [8]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [9]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5147017331855641,0,--
2000,1.4993392667924779,159,1:04
3000,1.4974250011972512,159,1:03
4000,1.494888994222998,158,1:03
5000,1.5030724921121883,157,1:03
6000,1.4983550320529422,157,1:02
7000,1.4946436860500825,157,1:01
8000,1.4934607363443555,157,1:01
9000,1.4966213503452224,156,1:01
10000,1.5006950138807684,155,1:01
11000,1.4919297555564408,153,1:01
12000,1.4969341033526733,151,1:01
13000,1.4951927822624822,149,1:01
14000,1.4943218759660055,149,1:01
15000,1.4898644709937254,150,1:00
16000,1.497972186073829,148,1:00
17000,1.5001833788468373,149,0:59
18000,1.4978046269765526,149,0:59
19000,1.4929294303339207,149,0:58
20000,1.4947190768716216,150,0:57
== Start Production ==
21000,1.4970225868654883,150,0:57
22000,1.4976996088863208,150,0:56
23000,1.4940191322837204,150,0:55
24000,1.4904828605065297,150,0:55
25000,1.4931573

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element EP found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 100/100 [00:03<00:00, 30.22it/s]


In [10]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)
trainer.neff = [50]

Initial neff: [30]


In [11]:
trainer.fit(10, 2)

  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:33<00:00,  2.96it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.83536061373131
Epoch 0 completed in 239.37 seconds.
Loss:  1.1026968717532737
Best Loss:  1.1026968717532737 at epoch  0


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:35<00:00,  2.83it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.835360626371305
Epoch 1 completed in 188.10 seconds.
Loss:  1.1026968717424739
Best Loss:  1.1026968717424739 at epoch  1


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:38<00:00,  2.59it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.83536056332045
Epoch 2 completed in 175.12 seconds.
Loss:  1.102696871756728
Best Loss:  1.1026968717424739 at epoch  1


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:31<00:00,  3.19it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.83536042996702
Epoch 3 completed in 162.02 seconds.
Loss:  1.1026968717859214
Best Loss:  1.1026968717424739 at epoch  1


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:30<00:00,  3.23it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.83536007705365
Epoch 4 completed in 160.46 seconds.
Loss:  1.1026968718504444
Best Loss:  1.1026968717424739 at epoch  1


In [12]:
trainer.neff = neff_prev

In [13]:
trainer.fit(490, 2)

  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:30<00:00,  3.28it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.8353598490317
Epoch 5 completed in 159.45 seconds.
Loss:  1.1026968718820536
Best Loss:  1.1026968717424739 at epoch  1


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:30<00:00,  3.33it/s]


Effective sample sizes:
  sample_0: 51
  Total: 51.835358688437175
Epoch 6 completed in 158.32 seconds.
Loss:  1.1026968721387327
Best Loss:  1.1026968717424739 at epoch  1


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:26<00:00,  3.70it/s]


...

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element EP found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 100/100 [00:01<00:00, 50.67it/s]


Epoch 196 completed in 226.95 seconds.
Loss:  1.0547418959903232
Best Loss:  1.021862120230441 at epoch  183


  0%|          | 0/100 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 100/100 [00:24<00:00,  4.02it/s]


Resampling sample_0... by xmlfiles/epoch-197.xml
Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.4924380056986692,0,--
2000,1.4915035379493728,174,0:58
3000,1.4795589368828341,172,0:58
4000,1.4877502759988377,171,0:58
5000,1.4818938127857264,170,0:58
6000,1.488512176204941,171,0:57
7000,1.479320146181475,171,0:57
8000,1.4860642718030643,172,0:56


OpenMMException: Particle coordinate is NaN.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#nan

In [ ]:
trainer.from_checkpoint(trainer_checkpoint="train_state.pkl",
                        ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
                        nums_ffxml=[2,1,1],
                        pdbfile="merged_supercell_bonds.pdb",
                        initial_ffxml="xmlfiles/epoch-4.xml",
                        loss_fn=lossfn,
                        opt_fftypes=["NonbondedForce/charges", 'VsiteForce/weight'],
                        )